# Iteration 1: GCN + 1D-CNN + fusion MLP

End-to-end link prediction on the NYC Bike Sharing dataset.

**Architecture**:

- **Graph branch (GCN)**: 2-layer GCN on a static graph. Edge weights = cumulative historical trip frequency in the training window. Node features = (capacity_norm, lat_norm, lon_norm, region_one_hot).
- **Node time-series branch (1D-CNN)**: 2-layer 1D-CNN over a 12-bin window (60 min) of the 4 availability series per station.
- **Fusion**: concat([gcn_u, gcn_v, cnn_u, cnn_v]) -> MLP -> sigmoid -> trip probability for the next 30 min.

**Data** (outside the vault): `C:/Users/user/Data/nyc-bike-sharing/{graph_edges.json, graph_nodes.json}`.

**Why these defaults**: see `Methoden-Bewertung.md` in the same folder.

## 0. Setup

If not installed yet, run the cell below once. PyTorch installs CPU-only, which is plenty for 2,213 nodes.

In [ ]:
# Run once in a fresh env, then comment out
%pip install --quiet torch pandas numpy scikit-learn ijson tqdm matplotlib


In [1]:
import json
import math
import time
import random
from dataclasses import dataclass
from pathlib import Path

import ijson
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


ModuleNotFoundError: No module named 'ijson'

## 1. Configuration

All hyperparameters in one place. Defaults come from the methods assessment. The first iteration is deliberately conservative.

In [ ]:
@dataclass
class Config:
    # Paths
    data_dir: Path = Path(r"C:/Users/user/Data/nyc-bike-sharing")

    # Time grid
    bin_minutes: int = 5
    window_bins: int = 12
    horizon_bins: int = 6
    sample_stride_bins: int = 6

    # Temporal split (data spans 2024-05-16 .. 2024-06-14)
    train_end: str = "2024-06-05T00:00:00"
    val_end:   str = "2024-06-09T00:00:00"

    # Sampling
    neg_per_pos: int = 5

    # Model
    static_hidden: int = 32
    static_out: int = 32
    cnn_channels: tuple = (16, 32)
    cnn_kernel: int = 3
    fusion_hidden: int = 64

    # Training
    batch_size: int = 256
    epochs: int = 10
    lr: float = 1e-3
    weight_decay: float = 1e-5

CFG = Config()
print(CFG)

## 2. Load the raw data

`graph_edges.json` fits fully in RAM. `graph_nodes.json` (530 MB) is streamed.

In [ ]:
t0 = time.time()
with open(CFG.data_dir / "graph_edges.json", "rb") as f:
    edges_raw = json.load(f)
print(f"edges loaded: {len(edges_raw)} super_edges in {time.time()-t0:.1f}s")


In [ ]:
t0 = time.time()
stations = []
ts_records = []

with open(CFG.data_dir / "graph_nodes.json", "rb") as f:
    for node in ijson.items(f, "item"):
        sid = node["station_id"]
        stations.append({
            "station_id": sid,
            "name": node["name"],
            "lat": float(node["lat"]),
            "lon": float(node["lon"]),
            "capacity": int(node["capacity"]),
            "region_id": node.get("region_id"),
        })
        for key, series in node["ts"].items():
            for rec in series:
                ts_records.append((sid, key, rec["Start"], int(rec["Value"])))

stations_df = pd.DataFrame(stations).set_index("station_id")
ts_df = pd.DataFrame(ts_records, columns=["station_id", "key", "Start", "Value"])
ts_df["Start"] = pd.to_datetime(ts_df["Start"])
print(f"nodes loaded: {len(stations_df)} stations, {len(ts_df):,} raw events in {time.time()-t0:.1f}s")


## 3. Index the stations

In [ ]:
station_ids = sorted(stations_df.index.unique())
sid_to_idx = {sid: i for i, sid in enumerate(station_ids)}
N = len(station_ids)
print(f"N stations: {N}")
stations_df = stations_df.loc[station_ids]
stations_df["idx"] = range(N)


## 4. Static node features

Four features: normalized capacity, normalized geo-coordinates, and region_id as one-hot. A missing `region_id` becomes its own category.

In [ ]:
def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / (s.std() + 1e-9)

stations_df["capacity_z"] = zscore(stations_df["capacity"])
stations_df["lat_z"] = zscore(stations_df["lat"])
stations_df["lon_z"] = zscore(stations_df["lon"])

region_filled = stations_df["region_id"].fillna("MISSING")
region_dummies = pd.get_dummies(region_filled, prefix="region").astype(np.float32)
stations_df = pd.concat([stations_df, region_dummies], axis=1)

feature_cols = ["capacity_z", "lat_z", "lon_z"] + list(region_dummies.columns)
X_static = torch.tensor(stations_df[feature_cols].values.astype(np.float32), dtype=torch.float32)
print(f"X_static shape: {tuple(X_static.shape)} (N x F_static)")


## 5. Static adjacency matrix

Edge weight per directed edge (u, v) = number of trips from u to v during the **training period**. This prevents leakage from val/test.

The adjacency matrix is symmetrized for the GCN convention. A directed variant would be iteration-2 material.

In [ ]:
train_end_dt = pd.to_datetime(CFG.train_end)

def rides_until(ts_series, cutoff):
    last_val = 0
    for rec in ts_series:
        if pd.to_datetime(rec["Start"]) < cutoff:
            last_val = rec["Value"]
        else:
            break
    return last_val

edge_rows = []
for e in edges_raw:
    if e["from"] not in sid_to_idx or e["to"] not in sid_to_idx:
        continue
    w = rides_until(e["ts"]["num_rides"], train_end_dt)
    if w == 0:
        continue
    u = sid_to_idx[e["from"]]
    v = sid_to_idx[e["to"]]
    edge_rows.append((u, v, w))

print(f"edges with trips during training: {len(edge_rows)} / {len(edges_raw)}")

A = torch.zeros(N, N, dtype=torch.float32)
for u, v, w in edge_rows:
    A[u, v] += w
    A[v, u] += w
A.diagonal().add_(1.0)  # +1 self-loop per node, GCN standard

deg = A.sum(dim=1)
D_inv_sqrt = torch.diag(1.0 / (deg.sqrt() + 1e-9))
A_norm = D_inv_sqrt @ A @ D_inv_sqrt
print("A_norm built:", A_norm.shape, "nonzeros:", (A_norm != 0).sum().item())


## 6. Resample the node time series

Put all four series onto a fixed 5-minute grid and forward-fill the gaps. Result: a tensor `X_ts` with shape `[N_stations, T_bins, 4_channels]`.

In [ ]:
global_start = pd.Timestamp("2024-05-16T00:00:00")
global_end   = pd.Timestamp("2024-06-14T00:00:00")
freq = f"{CFG.bin_minutes}min"
time_index = pd.date_range(global_start, global_end, freq=freq, inclusive="left")
T = len(time_index)
print(f"time grid: {T} bins of {CFG.bin_minutes} min = {T*CFG.bin_minutes/60:.1f} h")


In [ ]:
series_keys = ["num_bikes_available", "num_ebikes_available", "num_bikes_disabled", "num_docks_disabled"]
C_ts = len(series_keys)

X_ts = np.zeros((N, T, C_ts), dtype=np.float32)

ts_df_sorted = ts_df.sort_values(["station_id", "key", "Start"])
gb = ts_df_sorted.groupby(["station_id", "key"], sort=False)

t0 = time.time()
for (sid, key), grp in tqdm(gb, total=len(gb), desc="resample"):
    if sid not in sid_to_idx:
        continue
    if key not in series_keys:
        continue
    i = sid_to_idx[sid]
    c = series_keys.index(key)
    grp_series = pd.Series(grp["Value"].values, index=grp["Start"].values)
    grp_series = grp_series[~grp_series.index.duplicated(keep="last")]
    resampled = grp_series.reindex(time_index, method="ffill").fillna(0).astype(np.float32)
    X_ts[i, :, c] = resampled.values

print(f"resampling done in {time.time()-t0:.1f}s, X_ts shape {X_ts.shape}, dtype {X_ts.dtype}")


In [ ]:
ts_mean = X_ts.mean(axis=(0, 1), keepdims=True)
ts_std  = X_ts.std(axis=(0, 1), keepdims=True) + 1e-9
X_ts_norm = (X_ts - ts_mean) / ts_std
print("X_ts normalized:", X_ts_norm.shape, "mean ~0?", float(X_ts_norm.mean()))


## 7. Build the targets

For each directed edge (u, v) we compute delta num_rides over every 30-minute window. Label = 1 if delta > 0.

In [ ]:
def rides_timeline(ts_series, grid):
    starts = pd.to_datetime([r["Start"] for r in ts_series], format="ISO8601")  # robust against fractional seconds
    vals = np.array([r["Value"] for r in ts_series], dtype=np.int32)
    s = pd.Series(vals, index=starts)
    s = s[~s.index.duplicated(keep="last")]
    return s.reindex(grid, method="ffill").fillna(0).astype(np.int32).values

positive_samples = []
edge_horizon_diff = {}

H = CFG.horizon_bins
t0 = time.time()
for e in tqdm(edges_raw, desc="targets"):
    if e["from"] not in sid_to_idx or e["to"] not in sid_to_idx:
        continue
    u = sid_to_idx[e["from"]]
    v = sid_to_idx[e["to"]]
    timeline = rides_timeline(e["ts"]["num_rides"], time_index)
    diff = timeline[H:] - timeline[:-H]
    edge_horizon_diff[(u, v)] = diff
    pos_bins = np.where(diff > 0)[0]
    for tb in pos_bins:
        positive_samples.append((u, v, int(tb)))

print(f"positive samples: {len(positive_samples):,} in {time.time()-t0:.1f}s")


## 8. Temporal train/val/test split

Bin timestamp = start of the prediction window. We also filter on `t_bin >= window_bins` so the CNN has enough history.

In [ ]:
val_end_dt = pd.to_datetime(CFG.val_end)
train_end_idx = int((train_end_dt - global_start) // pd.Timedelta(minutes=CFG.bin_minutes))
val_end_idx   = int((val_end_dt   - global_start) // pd.Timedelta(minutes=CFG.bin_minutes))
test_end_idx  = T - H
print("train_end_idx", train_end_idx, "val_end_idx", val_end_idx, "test_end_idx", test_end_idx)

stride = CFG.sample_stride_bins
min_t = CFG.window_bins

def in_range(tb, lo, hi):
    return lo <= tb < hi and (tb % stride == 0)

train_pos = [s for s in positive_samples if in_range(s[2], min_t, train_end_idx)]
val_pos   = [s for s in positive_samples if in_range(s[2], train_end_idx, val_end_idx)]
test_pos  = [s for s in positive_samples if in_range(s[2], val_end_idx, test_end_idx)]
print(f"positives  train={len(train_pos):,}  val={len(val_pos):,}  test={len(test_pos):,}")


## 9. Model definition

Three small modules: GCN, 1D-CNN, fusion MLP. Deliberately minimal.

In [ ]:
class GCNBranch(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.W1 = nn.Linear(in_dim, hidden, bias=False)
        self.W2 = nn.Linear(hidden, out_dim, bias=False)
        self.drop = nn.Dropout(0.2)

    def forward(self, X, A_norm):
        H = A_norm @ self.W1(X)
        H = F.relu(H)
        H = self.drop(H)
        H = A_norm @ self.W2(H)
        return H


class CNN1DBranch(nn.Module):
    def __init__(self, in_channels=4, channels=(16, 32), kernel=3):
        super().__init__()
        c1, c2 = channels
        self.conv1 = nn.Conv1d(in_channels, c1, kernel_size=kernel, padding=kernel // 2)
        self.conv2 = nn.Conv1d(c1, c2, kernel_size=kernel, padding=kernel // 2)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.out_dim = c2

    def forward(self, x):
        # x: [B, T_window, C_in] -> [B, C_in, T_window]
        x = x.transpose(1, 2)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)
        return x


class LinkPredictor(nn.Module):
    def __init__(self, cfg, in_dim_static, c_ts):
        super().__init__()
        self.gcn = GCNBranch(in_dim_static, cfg.static_hidden, cfg.static_out)
        self.cnn = CNN1DBranch(in_channels=c_ts, channels=cfg.cnn_channels, kernel=cfg.cnn_kernel)
        fusion_in = 2 * cfg.static_out + 2 * self.cnn.out_dim
        self.mlp = nn.Sequential(
            nn.Linear(fusion_in, cfg.fusion_hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(cfg.fusion_hidden, 1),
        )

    def forward(self, X_static, A_norm, ts_u, ts_v, u_idx, v_idx):
        node_emb = self.gcn(X_static, A_norm)
        eu = node_emb[u_idx]
        ev = node_emb[v_idx]
        cu = self.cnn(ts_u)
        cv = self.cnn(ts_v)
        feat = torch.cat([eu, ev, cu, cv], dim=1)
        logits = self.mlp(feat).squeeze(-1)
        return logits


model = LinkPredictor(CFG, in_dim_static=X_static.shape[1], c_ts=C_ts).to(DEVICE)
print(model)


## 10. Batch sampler

Per batch: equal numbers of positive and negative examples (ratio 1 : neg_per_pos). Negatives are drawn at random from station pairs that have no positive entry for the same time window.

In [ ]:
positive_set = set((u, v, tb) for (u, v, tb) in positive_samples)


def sample_batch(positives, n_pos, neg_per_pos):
    pos = random.sample(positives, k=min(n_pos, len(positives)))
    samples = []
    labels = []
    for u, v, tb in pos:
        samples.append((u, v, tb))
        labels.append(1.0)
        for _ in range(neg_per_pos):
            while True:
                u2 = random.randrange(N)
                v2 = random.randrange(N)
                if u2 == v2:
                    continue
                if (u2, v2, tb) in positive_set:
                    continue
                break
            samples.append((u2, v2, tb))
            labels.append(0.0)
    return samples, labels


def batch_to_tensors(samples, labels):
    u_idx = torch.tensor([s[0] for s in samples], dtype=torch.long, device=DEVICE)
    v_idx = torch.tensor([s[1] for s in samples], dtype=torch.long, device=DEVICE)
    ts_u = torch.stack([
        torch.from_numpy(X_ts_norm[s[0], s[2] - CFG.window_bins:s[2]]) for s in samples
    ]).to(DEVICE)
    ts_v = torch.stack([
        torch.from_numpy(X_ts_norm[s[1], s[2] - CFG.window_bins:s[2]]) for s in samples
    ]).to(DEVICE)
    y = torch.tensor(labels, dtype=torch.float32, device=DEVICE)
    return u_idx, v_idx, ts_u, ts_v, y


## 11. Training

Loss: BCEWithLogitsLoss. Optimizer: Adam. One validation pass per epoch (AUC + AP).

In [ ]:
X_static_d = X_static.to(DEVICE)
A_norm_d = A_norm.to(DEVICE)

opt = torch.optim.Adam(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
loss_fn = nn.BCEWithLogitsLoss()


@torch.no_grad()
def evaluate(positives):
    model.eval()
    if not positives:
        return float("nan"), float("nan")
    samples, labels = sample_batch(positives, n_pos=len(positives), neg_per_pos=CFG.neg_per_pos)
    all_logits = []
    all_labels = []
    bs = 512
    for i in range(0, len(samples), bs):
        chunk = samples[i:i + bs]
        chunk_y = labels[i:i + bs]
        u_idx, v_idx, ts_u, ts_v, y = batch_to_tensors(chunk, chunk_y)
        logits = model(X_static_d, A_norm_d, ts_u, ts_v, u_idx, v_idx)
        all_logits.append(logits.cpu().numpy())
        all_labels.append(y.cpu().numpy())
    y = np.concatenate(all_labels)
    p = np.concatenate(all_logits)
    return roc_auc_score(y, p), average_precision_score(y, p)


steps_per_epoch = max(1, len(train_pos) // (CFG.batch_size // (CFG.neg_per_pos + 1)))
print("steps/epoch:", steps_per_epoch)

history = {"train_loss": [], "val_auc": [], "val_ap": []}
for epoch in range(1, CFG.epochs + 1):
    model.train()
    epoch_losses = []
    t0 = time.time()
    for step in range(steps_per_epoch):
        n_pos = CFG.batch_size // (CFG.neg_per_pos + 1)
        samples, labels = sample_batch(train_pos, n_pos=n_pos, neg_per_pos=CFG.neg_per_pos)
        u_idx, v_idx, ts_u, ts_v, y = batch_to_tensors(samples, labels)
        logits = model(X_static_d, A_norm_d, ts_u, ts_v, u_idx, v_idx)
        loss = loss_fn(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        epoch_losses.append(loss.item())
    val_auc, val_ap = evaluate(val_pos)
    history["train_loss"].append(float(np.mean(epoch_losses)))
    history["val_auc"].append(val_auc)
    history["val_ap"].append(val_ap)
    train_loss_str = f"{history['train_loss'][-1]:.4f}"
    print(f"epoch {epoch:2d}  loss {train_loss_str}  val AUC {val_auc:.4f}  val AP {val_ap:.4f}  ({time.time()-t0:.1f}s)")


## 12. Test-set evaluation

Final metrics on the test split: AUC, AP, and MRR@100 (1 positive sample against 99 random negatives).

In [ ]:
@torch.no_grad()
def evaluate_with_mrr(positives, n_neg_for_mrr=99):
    model.eval()
    auc, ap = evaluate(positives)
    rrs = []
    for u, v, tb in positives:
        negs = []
        while len(negs) < n_neg_for_mrr:
            u2 = random.randrange(N)
            v2 = random.randrange(N)
            if u2 == v2:
                continue
            if (u2, v2, tb) in positive_set:
                continue
            negs.append((u2, v2, tb))
        cands = [(u, v, tb)] + negs
        u_idx, v_idx, ts_u, ts_v, _ = batch_to_tensors(cands, [0.0] * len(cands))
        logits = model(X_static_d, A_norm_d, ts_u, ts_v, u_idx, v_idx).cpu().numpy()
        ranks = (-logits).argsort().argsort() + 1
        pos_rank = int(ranks[0])
        rrs.append(1.0 / pos_rank)
    mrr = float(np.mean(rrs)) if rrs else float("nan")
    return auc, ap, mrr


test_sample = random.sample(test_pos, k=min(500, len(test_pos)))
test_auc, test_ap, test_mrr = evaluate_with_mrr(test_sample)
print(f"TEST  AUC {test_auc:.4f}  AP {test_ap:.4f}  MRR@100 {test_mrr:.4f}  (on {len(test_sample)} positive samples)")


## 13. Learning curve

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(history["train_loss"], label="train loss")
ax[0].set_xlabel("epoch")
ax[0].set_title("Train Loss")
ax[0].grid(True)
ax[1].plot(history["val_auc"], label="val AUC")
ax[1].plot(history["val_ap"], label="val AP")
ax[1].set_xlabel("epoch")
ax[1].set_title("Validation")
ax[1].legend()
ax[1].grid(True)
plt.tight_layout()
plt.show()


## 14. Conclusion and next steps

What this iteration delivers:

- The end-to-end pipeline runs: data -> resampling -> graph -> model -> training -> evaluation.
- First baseline numbers for AUC, AP, MRR on the test split.
- Reproducibility via a fixed seed.

What was deliberately left out:

- No PyTorch Geometric; the GCN is implemented by hand.
- Static adjacency matrix (no time-varying graph topology).
- No time-based features (hour encoding, weekday).
- Negative sampling is purely random, no hard-negative mining.
- No early stopping, no learning-rate scheduling.

Planned for iteration 2 (see Methoden-Bewertung.md):

1. Graph-branch ablation: GCN -> GAT.
2. Time-series-branch ablation: 1D-CNN -> GRU.
3. Component ablation: graph only / time series only / no fusion MLP.
4. Sanity check `num_rides == classic_rides + electric_rides`.
5. Baselines: TGN, plus a spatiotemporal method from the Springer paper.